# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [2]:
from bs4 import BeautifulSoup

# Cargado en el notebook en la ruta /content/data/rotisserie-chicken.html
file = 'data/rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [3]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [4]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [5]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [6]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = []
for link in recipe_links:
    href = link['href']
    if "recipe" in href:
        recipe_urls.append(href)

# Print the recipe URLs
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Linked Recipes:
https://www.allrecipes.com/authentication/login?regSource=3675&relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
/account/add-recipe
https://www.myrecipes.com/favorites
https://www.allrecipes.com/authentication/logout?relativeRedirectUrl=%2Frecipe%2F93168%2Frotisserie-chicken%2F
https://www.magazines.com/allrecipes-magazine.html?utm_source=allrecipes.com&utm_medium=owned&utm_campaign=i111arr1w2661
https://www.magazines.com/allrecipes-magazine.html
https://www.allrecipes.com/recipes/17562/dinner/
https://www.allrecipes.com/recipes/17057/everyday-cooking/more-meal-ideas/5-ingredients/main-dishes/
https://www.allrecipes.com/recipes/15436/everyday-cooking/one-pot-meals/
https://www.allrecipes.com/recipes/1947/everyday-cooking/quick-and-easy/
https://www.allrecipes.com/recipes/455/everyday-cooking/more-meal-ideas/30-minute-meals/
https://www.allrecipes.com/recipes/17889/everyday-cooking/family-friendly/family-dinners/
https://www.allrecipes.com/recipes/94/soups-s

## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [13]:
import os
from bs4 import BeautifulSoup
import pandas as pd

# Path to the directory containing HTML files
data_dir = '/content/data/'

# Get a list of all HTML files in the directory
html_files = [f for f in os.listdir(data_dir) if f.endswith('.html')]

all_recipes_data = []

print(f"Procesando {len(html_files)} archivos HTML en {data_dir}\n")

# Iterate through each HTML file
for filename in html_files:
    file_path = os.path.join(data_dir, filename)

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            html_content = file.read()

        soup = BeautifulSoup(html_content, "html.parser")

        # --- Extracting information ---

        # Title
        title_tag = soup.find("meta", {"property": "og:title"})
        title = title_tag["content"] if title_tag else "N/A"

        # Description
        description_tag = soup.find("meta", {"name": "description"})
        description = description_tag["content"] if description_tag else "N/A"

        # Ingredients
        ingredients_list = []
        ingredients_section_1 = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
        ingredients_section_2 = soup.find_all("span", class_="ingredients-item-name") # Fallback

        if ingredients_section_1:
            for item in ingredients_section_1:
                text = item.get_text().strip()
                if text and text.lower() not in ["add to cart", "unidad", "units"]:
                    ingredients_list.append(text)
        elif ingredients_section_2:
            for item in ingredients_section_2:
                text = item.get_text().strip()
                if text and text.lower() not in ["add to cart", "unidad", "units"]:
                    ingredients_list.append(text)

        # Instructions
        instructions_list = []
        instructions_elements_1 = soup.find_all("li", class_="comp mntl-sc-block mntl-sc-block-group--LI mntl-sc-block instruction-section__item")
        instructions_elements_2 = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html") # Fallback

        if instructions_elements_1:
            for instr in instructions_elements_1:
                text = instr.get_text().strip()
                if text and "Ingredients:" not in text and "Directions:" not in text:
                    instructions_list.append(text)
        elif instructions_elements_2:
            for instr in instructions_elements_2:
                text = instr.get_text().strip()
                # Ensure we only pick up actual instructions, avoiding boilerplate or headers
                if text and "Ingredients:" not in text and "Directions:" not in text and len(text) > 10:
                    instructions_list.append(text)

        # Nutrition Facts
        nutrition_facts_list = []
        nutrition_section_1 = soup.find_all("div", class_="mntl-nutrition-facts-label__table-body")
        nutrition_section_2 = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix") # Fallback

        if nutrition_section_1:
            # Extract from the structured table body
            for item in nutrition_section_1[0].find_all(['div', 'span'], class_=lambda x: x != 'mntl-sr-only'):
                text = item.get_text(separator=' ', strip=True)
                if text and text not in ["Amount Per Serving", "Datos Nutricionales", "Daily Value"]: # Filter common headers
                    nutrition_facts_list.append(text)
        elif nutrition_section_2:
            # Extract from a list of nutrient names and their values
            for fact in nutrition_section_2:
                parent_text = fact.parent.get_text(separator=' ', strip=True)
                nutrition_facts_list.append(parent_text)

        recipe_data = {
            "Recipe Title": title,
            "Description": description,
            "Ingredients": "\n-".join(ingredients_list) if ingredients_list else "N/A",
            "Instruction": "\n ".join([f"{i+1}. {instr}" for i, instr in enumerate(instructions_list)]) if instructions_list else "N/A",
            "Nutrition Facts": "; ".join(nutrition_facts_list) if nutrition_facts_list else "N/A"
        }
        all_recipes_data.append(recipe_data)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# Create DataFrame
if all_recipes_data:
    recipes_df = pd.DataFrame(all_recipes_data)
    pd.set_option('display.max_colwidth', None) # Display full content in columns
    print("DataFrame de Recetas Rescatadas:")
    display(recipes_df)
else:
    print("No se pudieron extraer datos de recetas de los archivos HTML.")

Procesando 16 archivos HTML en /content/data/

DataFrame de Recetas Rescatadas:


Recipe Title  \
0                Darn Good Chicken   
1    Good Frickin’ Paprika Chicken   
2             Smoked Whole Chicken   
3            Best Beer Can Chicken   
4      Buttermilk Barbecue Chicken   
5                    Drunk Chicken   
6    Grilled Chicken Under a Brick   
7            Easy Barbeque Chicken   
8         Smoked Beer Butt Chicken   
9               Rotisserie Chicken   
10  The Best Beer Can Chicken Ever   
11              Miso Honey Chicken   
12   Cilantro-Lime Grilled Chicken   
13     Rosemary Buttermilk Chicken   
14    Grilled Spatchcocked Chicken   
15               Beer Butt Chicken   

                                                                                                                                               Description  \
0                                                                                            Nutmeg offers this quick honey mustard sauce extra sweetness.   
1    Chef John named his delicious yogurt- and paprika-marinated grilled chicken as an homage to one of the best chicken take-out joints in San Francisco.   
2                                                 Spatchcocked and rubbed with a spicy seasoning blend, this whole smoked chicken is bursting with flavor.   
3   In this best beer can chicken recipe, chicken is seasoned with a sweet and spicy brown sugar-paprika rub and grilled over wood chips for smoky flavor.   
4                  Chicken turns out super moist and flavorful on the grill after soaking in Chef John's subtly smoky, slightly sweet buttermilk marinade.   
5                                                              Drunken chicken is cooked over the grill with a beer can up inside for a fun and easy dish!   
6                   This is a very basic recipe for chicken on a brick. Feel free to brine the chicken beforehand or season it with extra herbs or spices.   
7    Grilled chicken bursts with flavor after a bath in a simple marinade. Oregano is the secret ingredient in this recipe - increase the amount to taste!   
8            Roast a chicken on your grill with a can of beer flavored with garlic and liquid smoke flavoring for a smoky and tender chicken they'll love.   
9     Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.   
10       A whole grilled chicken gets a burst of flavor from a warm and complex spice rub and a can of dark stout beer with hot peppers and garlic cloves.   
11   Chicken is marinated in Chef John's magical miso-honey sauce and grilled over indirect heat to make this simple yet umami-packed miso chicken recipe.   
12        This cilantro-lime grilled chicken recipe starts with a quick four-ingredient marinade with lime juice, cilantro, garlic salt, and black pepper.   
13             Summer is the time to master this garlicky buttermilk chicken grilling recipe that is simply infused with fresh rosemary and smoky paprika.   
14   This grilled spatchcock chicken recipe calls for removing the chicken's backbone and spreading it open like a book for quicker and more even cooking.   
15            This beer butter chicken recipe combines beer, butter, chicken, and seasonings for a moist and flavorful meal that will have people talking.   

                                                                                                                                                                                                                                                                                                                                                                                                                        Ingredients  \
0                                                                                                                                                                                                                                                                        

In [16]:
!pip install -U sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.3 MB/s eta 0:00:00


In [18]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Asegúrate de que `recipes_df` esté disponible desde la celda anterior.
# Si no lo está, esta parte del código te avisará.
if 'recipes_df' not in locals():
    print("Error: 'recipes_df' no está disponible. Por favor, ejecuta la celda anterior para crear el DataFrame de recetas.")
else:
    # 1. Preparar el corpus RAG a partir del DataFrame
    corpus_documents = []
    for index, row in recipes_df.iterrows():
        doc_text = f"""Recipe Title: {row['Recipe Title']}
Description: {row['Description']}

Ingredients:
{row['Ingredients']}

Instructions:
{row['Instruction']}

Nutrition Facts:
{row['Nutrition Facts']}"""
        corpus_documents.append(doc_text)

    print(f"Corpus RAG creado con {len(corpus_documents)} documentos (recetas).\n")

    # Cargar un modelo de embeddings pre-entrenado
    # 'all-MiniLM-L6-v2' es un buen modelo equilibrado para muchas tareas.
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Crear embeddings para los documentos del corpus
    print("--- Creando Embeddings del Corpus ---")
    corpus_embeddings = model.encode(corpus_documents, convert_to_tensor=True)
    corpus_embeddings_np = corpus_embeddings.cpu().numpy().astype('float32')

    # 3. Configurar un Índice FAISS para búsqueda eficiente
    dimension = corpus_embeddings_np.shape[1] # Dimensión de los vectores de embedding
    index = faiss.IndexFlatL2(dimension) # Usamos IndexFlatL2 para búsqueda de distancia euclidiana
    index.add(corpus_embeddings_np) # Añadir los embeddings al índice

    print(f"Tamaño del Corpus: {len(corpus_documents)} documentos")
    print(f"Dimensión del Embedding: {dimension}")
    print(f"Índice FAISS creado con {index.ntotal} vectores.\n")

    # 4. Implementar la función de recuperación (Retrieval)
    def retrieve_documents(query, k=1):
        """
        Busca los documentos más relevantes en el corpus dada una consulta.
        Args:
            query (str): La consulta del usuario.
            k (int): Número de documentos a recuperar.
        Returns:
            list: Una lista de los documentos más relevantes.
        """
        # Crear el embedding para la consulta
        query_embedding = model.encode([query], convert_to_tensor=True)
        query_embedding_np = query_embedding.cpu().numpy().astype('float32')

        # Buscar en el índice FAISS
        distances, indices = index.search(query_embedding_np, k)

        retrieved_docs = []
        for i in range(k):
            doc_index = indices[0][i]
            retrieved_docs.append(corpus_documents[doc_index])
        return retrieved_docs

    # --- Demostración de uso de la recuperación ---
    print("--- Realizando una Búsqueda RAG (Fase de Recuperación) ---")
    query_rag = "¿Cuáles son los ingredientes principales del pollo asado con cerveza y cómo se prepara?"
    retrieved_content = retrieve_documents(query_rag, k=1)

    print(f"Consulta: '{query_rag}'")
    print("\nContenido Recuperado (primeros 500 caracteres del documento más relevante):")
    for doc in retrieved_content:
        print(doc[:500] + "...")

Corpus RAG creado con 16 documentos (recetas).



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- Creando Embeddings del Corpus ---
Tamaño del Corpus: 16 documentos
Dimensión del Embedding: 384
Índice FAISS creado con 16 vectores.

--- Realizando una Búsqueda RAG (Fase de Recuperación) ---
Consulta: '¿Cuáles son los ingredientes principales del pollo asado con cerveza y cómo se prepara?'

Contenido Recuperado (primeros 500 caracteres del documento más relevante):
Recipe Title: Good Frickin’ Paprika Chicken
Description: Chef John named his delicious yogurt- and paprika-marinated grilled chicken as an homage to one of the best chicken take-out joints in San Francisco.

Ingredients:
6 tablespoons plain yogurt
-3 cloves garlic, crushed
-3 tablespoons ground paprika
-2 tablespoons olive oil
-1 tablespoon hot chile paste (such as sambal oelek)
-1 pinch cayenne pepper
-1 (5 pound) whole chicken, cut into 8 pieces
-salt
-¼ cup olive oil
-2 tablespoons sherry vin...
